# EnzAIme v2: stability and structural-image experiments
Attach enzaime_v2_training.zip as a private Kaggle dataset. This package already contains curated public Tm records and 13 sequence-verified structure images. Enable Internet and T4 x2. Run in order. No model in this notebook is a validated mutation-effect oracle.


In [ ]:
from pathlib import Path
import os, shutil, zipfile, sys, subprocess, json
root = Path('/kaggle/working/EnzAIme_v2')
inputs = Path('/kaggle/input')
sources = [p.parent.parent for p in inputs.rglob('train_stability_oracle.py')
           if (p.parent.parent/'data/stability/records.json').is_file()]
archives = list(inputs.rglob('enzaime_v2_training.zip'))
root.mkdir(parents=True, exist_ok=True)
if len(sources) == 1:
    shutil.copytree(sources[0], root, dirs_exist_ok=True)
elif not sources and len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as z:
        assert all((root/n).resolve().is_relative_to(root.resolve()) for n in z.namelist())
        z.extractall(root)
else:
    raise RuntimeError(f'Attach exactly one v2 dataset. Found roots: {sources}; ZIPs: {archives}')
os.chdir(root)
print('Ready:', root)
print(json.loads(Path('data/stability/manifest.json').read_text()))


In [ ]:
%pip install -r backend/requirements-neural.txt


In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable GPU in notebook settings'
print('GPUs:', torch.cuda.device_count())
env_activity = dict(os.environ, CUDA_VISIBLE_DEVICES='0')
env_stability = dict(os.environ, CUDA_VISIBLE_DEVICES='1' if torch.cuda.device_count() > 1 else '0')
logs = Path('training_logs'); logs.mkdir(exist_ok=True)
# Start Tm training on the second T4 while activity/image experiments use the first.
stability_log = (logs/'stability.log').open('w')
stability_process = subprocess.Popen([sys.executable, '-u', '-m', 'scripts.train_stability_oracle'],
    env=env_stability, stdout=stability_log, stderr=subprocess.STDOUT)
print('Stability job started; progress in training_logs/stability.log')


In [ ]:
subprocess.run([sys.executable, '-m', 'scripts.prepare_neural_features',
    '--images', 'data/structure_images/protein_views.json', '--output', 'data/neural_features_v2'], env=env_activity, check=True)
subprocess.run([sys.executable, '-m', 'scripts.train_neural_oracle',
    '--features', 'data/neural_features_v2', '--sequence-only', '--output', 'models/neural_sequence'], env=env_activity, check=True)
subprocess.run([sys.executable, '-m', 'scripts.train_neural_oracle',
    '--features', 'data/neural_features_v2', '--output', 'models/neural_images'], env=env_activity, check=True)


## Wait for the stability checkpoint
Keep this cell running until the second job finishes. Failure stops packaging. Read logs rather than restarting duplicate GPU processes.


In [ ]:
import time
while stability_process.poll() is None:
    print('\n'.join((logs/'stability.log').read_text(errors='replace').splitlines()[-3:]), flush=True)
    time.sleep(30)
stability_log.close()
if stability_process.returncode != 0:
    print((logs/'stability.log').read_text(errors='replace')[-10000:])
    raise RuntimeError('Stability training failed; resolve the logged error before exporting')
for folder in ['neural_sequence','neural_images','stability']:
    manifest = json.loads(Path(f'models/{folder}/manifest.json').read_text())
    print(folder, manifest['test_metrics'])


## Return the complete experiment
Return all three checkpoints, features and logs. Compare image and sequence models on the same split; do not choose repeatedly on the held-out test set. The current local model is unchanged until these experiments are reviewed.


In [ ]:
output = Path('/kaggle/working/enzaime_v2_results.zip')
with zipfile.ZipFile(output, 'w', zipfile.ZIP_DEFLATED) as archive:
    for folder in ['models/neural_sequence','models/neural_images','models/stability','data/neural_features_v2','training_logs']:
        for file in Path(folder).rglob('*'):
            if file.is_file(): archive.write(file, str(file))
print('Download:', output)
